In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Reload the dataframe and dataset class from Day 1
df = pd.read_csv("../data/HAM10000_metadata.csv")

import os, glob
part1 = glob.glob("../data/HAM10000_images_part_1/*.jpg")
part2 = glob.glob("../data/HAM10000_images_part_2/*.jpg")
all_images = part1 + part2
image_paths = {os.path.splitext(os.path.basename(p))[0]: p for p in all_images}
df['image_path'] = df['image_id'].map(image_paths)
df = df.dropna(subset=['image_path'])
print(f"Dataset ready: {len(df)} images")

Using device: cpu
Dataset ready: 10015 images


In [8]:
# Training transform — augment to help generalization
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Validation/test transform — no augmentation, just resize
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

print("Transforms defined ✅")

Transforms defined ✅


In [9]:
from torch.utils.data import Dataset
from PIL import Image

class HAMDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {
            'nv': 0, 'mel': 1, 'bkl': 2,
            'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6
        }
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.label_map[row['dx']]

# Split: 70% train, 15% val, 15% test
total = len(df)
train_size = int(0.70 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size + val_size]
test_df = df.iloc[train_size + val_size:]

train_dataset = HAMDataset(train_df, transform=train_transform)
val_dataset = HAMDataset(val_df, transform=val_transform)
test_dataset = HAMDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 7010 | Val: 1502 | Test: 1503


In [10]:
from collections import Counter

label_map = {'nv':0,'mel':1,'bkl':2,'bcc':3,'akiec':4,'vasc':5,'df':6}
label_counts = Counter(train_df['dx'].map(label_map))
total_train = len(train_df)

class_weights = torch.zeros(7)
for cls, count in label_counts.items():
    class_weights[cls] = total_train / (7 * count)

class_weights = class_weights.to(device)
print("Class weights:", class_weights)
print("Higher weight = rarer class = model pays more attention to it ✅")

Class weights: tensor([0.2482, 0.9022, 0.9145, 1.9483, 0.0000, 7.0523, 8.7081])
Higher weight = rarer class = model pays more attention to it ✅


In [11]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += out.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss/len(loader), 100.*correct/total

def validate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            total_loss += criterion(out, labels).item()
            correct += out.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    return total_loss/len(loader), 100.*correct/total

print("Starting training — 5 epochs to verify model is learning...\n")
for epoch in range(5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    vl_loss, vl_acc = validate(model, val_loader)
    print(f"Epoch {epoch+1}/5 | "
          f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.1f}% | "
          f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc:.1f}%")

Starting training — 5 epochs to verify model is learning...



RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x8192 and 2048x256)

In [12]:
# What you should see:
# - Train loss going DOWN each epoch ✅
# - Train accuracy going UP each epoch ✅
# - Val accuracy somewhere between 40-60% after 5 epochs is normal
# - If train loss is NOT going down after 3 epochs, tell me

print("✅ If train loss is decreasing — model is learning correctly")
print("✅ Val accuracy 40-60% after 5 epochs on HAM10000 is normal and expected")
print("⚠️  We'll train for 20+ epochs with early stopping on Day 4")
print("\nDay 2 done. Commit and rest.")

✅ If train loss is decreasing — model is learning correctly
✅ Val accuracy 40-60% after 5 epochs on HAM10000 is normal and expected
⚠️  We'll train for 20+ epochs with early stopping on Day 4

Day 2 done. Commit and rest.
